# Industrial Defect Detection: Metal Surface Defect Classification Model
## Phase 2 CNN Project - End-to-End Implementation

This notebook contains the complete, self-contained deep learning pipeline to load, preprocess, augment, train, evaluate, and interpret classification models for hot-rolled steel surface defects using the **NEU Metal Surface Defect Dataset**.

### Project Outline:
1. **Setup & Path Configurations**: Locate and verify dataset directories.
2. **Exploratory Data Analysis (EDA)**: Plot class distribution and display defect sample images.
3. **Preprocessing & Augmentation Pipeline**: Stratified 70/15/15 splitting and augmentation visualization.
4. **Custom CNN Training**: Designing and training a convolutional network from scratch.
5. **MobileNetV2 Transfer Learning**: Frozen feature extraction and subsequent fine-tuning.
6. **Evaluation & Visualization**: Generating comparative training curves, confusion matrices, and ROC curves.
7. **Interpretability via Grad-CAM**: Visualizing defect-specific neural network activations.
8. **Error Analysis**: Displaying correct vs incorrect predictions.
9. **Model Comparison**: Creating and saving the final performance comparison table.

In [ ]:
import os
import time
import glob
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Input, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
print("Libraries loaded. GPU Available:", tf.config.list_physical_devices('GPU'))

### 1. Setup & Path Configurations
Define directories containing the dataset. This notebook searches multiple default directories to run out-of-the-box on Kaggle or a local machine.

In [ ]:
# Search paths for Kaggle environments or local repositories
possible_dirs = [
    '/kaggle/input/neu-surface-defect-database/NEU-DET',
    '/kaggle/input/neu-surface-defect-database/NEU-DET/images',
    '/kaggle/input/neu-surface-defect-database',
    '../data/raw',
    './data/raw'
]

DATA_DIR = None
for path in possible_dirs:
    if os.path.exists(path) and os.path.isdir(path):
        # Ensure it has class directories inside or files
        DATA_DIR = path
        break

if DATA_DIR is None:
    print("WARNING: Dataset directory not found automatically. Setting default to './data/raw'.")
    DATA_DIR = './data/raw'
else:
    print(f"Dataset found at: {DATA_DIR}")

# Create outputs directories
os.makedirs('outputs/figures', exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)

### 2. Exploratory Data Analysis (EDA)
Examine dataset distribution and display sample images from each category.

In [ ]:
# Walk directory and list classes
extensions = ('*.bmp', '*.jpg', '*.jpeg', '*.png', '*.BMP', '*.JPG', '*.JPEG', '*.PNG')
image_paths = []
for ext in extensions:
    image_paths.extend(glob.glob(os.path.join(DATA_DIR, '**', ext), recursive=True))

if not image_paths:
    print(f"No image files found in '{DATA_DIR}'. Please check the dataset files.")
else:
    data_records = []
    for path in image_paths:
        label = os.path.basename(os.path.dirname(path))
        data_records.append({'filepath': path, 'label': label})
    df = pd.DataFrame(data_records)

    # Print class distribution
    print(f"Total Images Found: {len(df)}")
    print(df['label'].value_counts())
    
    # Plot class balance chart
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x='label', palette='viridis')
    plt.title('NEU Dataset Class Distribution')
    plt.xlabel('Defect Class')
    plt.ylabel('Count')
    plt.xticks(rotation=15)
    plt.savefig('outputs/figures/eda_class_distribution.png', bbox_inches='tight')
    plt.show()

    # Display sample images from each class
    classes = df['label'].unique()
    plt.figure(figsize=(12, 8))
    for idx, label in enumerate(sorted(classes)):
        sample_path = df[df['label'] == label]['filepath'].values[0]
        img = cv2.imread(sample_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        plt.subplot(2, 3, idx + 1)
        plt.imshow(img)
        plt.title(f"{label}\nShape: {img.shape[:2]}")
        plt.axis('off')
    plt.suptitle("NEU Dataset Defect Samples", fontsize=14, weight='bold')
    plt.tight_layout()
    plt.savefig('outputs/figures/eda_defect_samples.png', bbox_inches='tight')
    plt.show()

### 3. Preprocessing & Augmentation Pipeline
Perform a stratified split: **70% training, 15% validation, and 15% testing**. Then define generators with augmentations.

In [ ]:
if 'df' in locals():
    # Stratified Train/Val/Test splitting
    train_df, temp_df = train_test_split(
        df, test_size=0.30, random_state=42, stratify=df['label']
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
    )

    print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")
    
    TARGET_SIZE = (224, 224)
    BATCH_SIZE = 32

    # Data augmentations for training data
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.2,
        brightness_range=[0.8, 1.2],
        horizontal_flip=True,
        vertical_flip=True,
        fill_mode='nearest'
    )

    # Val/Test: only rescale
    val_datagen = ImageDataGenerator(rescale=1./255)
    test_datagen = ImageDataGenerator(rescale=1./255)

    # Define Generators
    train_gen = train_datagen.flow_from_dataframe(
        train_df, x_col='filepath', y_col='label', target_size=TARGET_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True, seed=42
    )
    val_gen = val_datagen.flow_from_dataframe(
        val_df, x_col='filepath', y_col='label', target_size=TARGET_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False, seed=42
    )
    test_gen = test_datagen.flow_from_dataframe(
        test_df, x_col='filepath', y_col='label', target_size=TARGET_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False, seed=42
    )
    
    # Augmentation visualization cell
    plt.figure(figsize=(10, 4))
    img_path = train_df.iloc[0]['filepath']
    img = load_img(img_path, target_size=TARGET_SIZE)
    x = img_to_array(img) / 255.0
    x = np.expand_dims(x, axis=0)
    
    idx = 1
    for batch in train_datagen.flow(x, batch_size=1):
        plt.subplot(1, 4, idx)
        plt.imshow(batch[0])
        plt.title(f"Augmented {idx}")
        plt.axis('off')
        idx += 1
        if idx > 4:
            break
    plt.suptitle("Sample Real-time Training Data Augmentations", weight='bold')
    plt.tight_layout()
    plt.savefig('outputs/figures/preprocessing_augmentation_samples.png', bbox_inches='tight')
    plt.show()

### 4. Custom CNN Model
Design and train the Custom CNN architecture from scratch.

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=6):
    model = Sequential([
        Input(shape=input_shape),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ], name="Custom_CNN")
    return model

custom_model = build_custom_cnn()
custom_model.summary()

custom_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_custom = [
    ModelCheckpoint('outputs/models/custom_cnn.keras', monitor='val_accuracy', save_best_only=True, mode='max'),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4, min_lr=1e-6)
]

print("Starting Custom CNN training...")
start_time = time.time()
history_custom = custom_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=12,  # Set lower epoch limit for faster notebook execution
    callbacks=callbacks_custom
)
custom_train_time = time.time() - start_time
print(f"Custom CNN training finished in {custom_train_time:.2f} seconds.")

### 5. Transfer Learning (MobileNetV2)
Implement Transfer Learning with pre-trained MobileNetV2, followed by a fine-tuning stage.

In [ ]:
# Load pre-trained base model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze layers initially

# Add classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(6, activation='softmax')(x)

mobile_model = Model(inputs=base_model.input, outputs=predictions, name="MobileNetV2_Transfer")
mobile_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_mob = [
    ModelCheckpoint('outputs/models/mobilenetv2.keras', monitor='val_accuracy', save_best_only=True, mode='max'),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
]

print("Starting MobileNetV2 Feature Extraction training...")
start_time_mob = time.time()
history_mobile = mobile_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks_mob
)

# 5.2 Fine-Tuning Phase
print("Unfreezing deep layers for fine-tuning...")
base_model.trainable = True
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

mobile_model.compile(
    optimizer=Adam(learning_rate=1e-5), # Lower learning rate for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Starting MobileNetV2 Fine-Tuning...")
history_fine = mobile_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=callbacks_mob
)
mobile_train_time = time.time() - start_time_mob
print(f"MobileNetV2 total training finished in {mobile_train_time:.2f} seconds.")

# Merge histories
merged_history = {}
for key in history_mobile.history.keys():
    merged_history[key] = history_mobile.history[key] + history_fine.history[key]

### 6. Performance Evaluation & Visualization
Plot training curves, evaluate models on the holdout test set, generate confusion matrices, and plot ROC curves.

In [ ]:
# 6.1 Plot training curves
def plot_curves(history, title, save_name):
    acc = history['accuracy']
    val_acc = history['val_accuracy']
    loss = history['loss']
    val_loss = history['val_loss']
    epochs = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, acc, 'b-', label='Train Acc')
    ax1.plot(epochs, val_acc, 'r-', label='Val Acc')
    ax1.set_title(f'{title} Accuracy')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(epochs, loss, 'b-', label='Train Loss')
    ax2.plot(epochs, val_loss, 'r-', label='Val Loss')
    ax2.set_title(f'{title} Loss')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(f'outputs/figures/{save_name}.png', bbox_inches='tight')
    plt.show()

plot_curves(history_custom.history, 'Custom CNN', 'custom_learning_curves')
plot_curves(merged_history, 'MobileNetV2', 'mobilenetv2_learning_curves')

In [ ]:
# 6.2 Test Set Evaluation
class_names = list(test_gen.class_indices.keys())
y_true = test_gen.classes

def evaluate_on_test(model, name):
    test_gen.reset()
    preds = model.predict(test_gen)
    y_pred = np.argmax(preds, axis=1)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    print(f"=== {name} Classification Report ===")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {name}')
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')
    plt.tight_layout()
    plt.savefig(f'outputs/figures/confusion_matrix_{name.lower().replace(" ", "_")}.png', bbox_inches='tight')
    plt.show()
    
    return acc, prec, rec, f1, preds

custom_metrics = evaluate_on_test(custom_model, "Custom CNN")
mobile_metrics = evaluate_on_test(mobile_model, "MobileNetV2")

In [ ]:
# 6.3 Multiclass ROC Curves
def plot_multiclass_roc(y_true, preds, name):
    y_true_bin = to_categorical(y_true, num_classes=6)
    fpr = {}
    tpr = {}
    roc_auc = {}
    
    for i in range(6):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], preds[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        
    plt.figure(figsize=(7, 5))
    for i, cls in enumerate(class_names):
        plt.plot(fpr[i], tpr[i], label=f"{cls} (AUC = {roc_auc[i]:.2f})")
        
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'One-vs-Rest ROC Curves - {name}')
    plt.legend(loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f'outputs/figures/roc_curves_{name.lower().replace(" ", "_")}.png', bbox_inches='tight')
    plt.show()

plot_multiclass_roc(y_true, custom_metrics[4], "Custom CNN")
plot_multiclass_roc(y_true, mobile_metrics[4], "MobileNetV2")

### 7. Explainable AI: Grad-CAM Visualizations
Implement Grad-CAM to overlay neural activation heatmaps on raw defect images, ensuring that the classifications align with the physical coordinates of the steel surface defects.

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        inputs=[model.inputs],
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0.0)
    max_val = tf.math.reduce_max(heatmap)
    if max_val == 0:
        max_val = 1e-10
    heatmap = heatmap / max_val
    return heatmap.numpy()

def overlay_gradcam(img_path, heatmap, alpha=0.4):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    heatmap = np.uint8(255 * heatmap)
    jet = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    jet = cv2.resize(jet, (img.shape[1], img.shape[0]))
    jet = cv2.cvtColor(jet, cv2.COLOR_BGR2RGB)
    
    superimposed_img = cv2.addWeighted(jet, alpha, img, 1 - alpha, 0)
    return img, superimposed_img

# Find last conv layers dynamically
def get_last_conv_name(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    return None

custom_conv = get_last_conv_name(custom_model)
# For MobileNetV2, find the convolutional layer in its architecture
mobile_conv = None
for layer in reversed(mobile_model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D) or 'conv' in layer.name.lower():
        mobile_conv = layer.name
        break

print(f"Custom CNN conv: {custom_conv} | MobileNetV2 conv: {mobile_conv}")

# Display Grad-CAM on a test image
if len(test_df) > 0 and custom_conv and mobile_conv:
    test_sample = test_df.iloc[0]
    img_path = test_sample['filepath']
    
    img_load = load_img(img_path, target_size=TARGET_SIZE)
    img_arr = img_to_array(img_load) / 255.0
    img_arr = np.expand_dims(img_arr, axis=0)
    
    # Generate heatmaps
    custom_heatmap = make_gradcam_heatmap(img_arr, custom_model, custom_conv)
    mobile_heatmap = make_gradcam_heatmap(img_arr, mobile_model, mobile_conv)
    
    orig, custom_cam = overlay_gradcam(img_path, custom_heatmap)
    _, mobile_cam = overlay_gradcam(img_path, mobile_heatmap)
    
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(orig)
    plt.title(f"Original Defect\n(Class: {test_sample['label']})")
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(custom_cam)
    plt.title("Custom CNN Grad-CAM")
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.imshow(mobile_cam)
    plt.title("MobileNetV2 Grad-CAM")
    plt.axis('off')
    
    plt.suptitle("Explainable AI: Defect Heatmap Activations", fontsize=13, weight='bold')
    plt.tight_layout()
    plt.savefig('outputs/figures/gradcam_comparison_sample.png', bbox_inches='tight')
    plt.show()

### 8. Error Analysis
Identify where models predict correctly and where they make mistakes to pinpoint potential improvements.

In [ ]:
test_gen.reset()
preds_mob = mobile_model.predict(test_gen)
y_pred_mob = np.argmax(preds_mob, axis=1)

correct_indices = np.where(y_pred_mob == y_true)[0]
incorrect_indices = np.where(y_pred_mob != y_true)[0]

print(f"Out of {len(y_true)} test samples:")
print(f"  Correctly predicted: {len(correct_indices)}")
print(f"  Incorrectly predicted: {len(incorrect_indices)}")

# Plot Correct vs Incorrect Predictions
plt.figure(figsize=(10, 5))

# Display 2 correct samples
for i in range(min(2, len(correct_indices))):
    idx = correct_indices[i]
    path = test_df.iloc[idx]['filepath']
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(1, 4, i + 1)
    plt.imshow(img)
    plt.title(f"Correct\nTrue: {class_names[y_true[idx]]}\nPred: {class_names[y_pred_mob[idx]]}", color='green')
    plt.axis('off')

# Display 2 incorrect samples (if any)
for i in range(min(2, len(incorrect_indices))):
    idx = incorrect_indices[i]
    path = test_df.iloc[idx]['filepath']
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(1, 4, i + 3)
    plt.imshow(img)
    plt.title(f"Error\nTrue: {class_names[y_true[idx]]}\nPred: {class_names[y_pred_mob[idx]]}", color='red')
    plt.axis('off')

plt.suptitle("Error Analysis: Sample Predictions", fontsize=13, weight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/error_analysis_samples.png', bbox_inches='tight')
plt.show()

### 9. Model Comparison
Compile metrics and training durations for both architectures and save the summary table as a high-resolution PNG image.

In [ ]:
# Construct Comparison DataFrame
comparison_data = {
    'Metric': ['Accuracy', 'Precision (Weighted)', 'Recall (Weighted)', 'F1 Score (Weighted)', 'Training Time'],
    'Custom CNN': [
        f"{custom_metrics[0]:.2%}", f"{custom_metrics[1]:.4f}", f"{custom_metrics[2]:.4f}", f"{custom_metrics[3]:.4f}", f"{custom_train_time/60:.2f} min"
    ],
    'MobileNetV2': [
        f"{mobile_metrics[0]:.2%}", f"{mobile_metrics[1]:.4f}", f"{mobile_metrics[2]:.4f}", f"{mobile_metrics[3]:.4f}", f"{mobile_train_time/60:.2f} min"
    ]
}

comp_df = pd.DataFrame(comparison_data)
print("=== Performance Comparison Table ===")
print(comp_df)

# Render comparative table as matplotlib figure
fig, ax = plt.subplots(figsize=(8, 3), dpi=300)
ax.axis('off')
ax.axis('tight')

table = ax.table(
    cellText=comp_df.values, 
    colLabels=comp_df.columns, 
    loc='center', 
    cellLoc='center',
    colColours=['#4F81BD', '#4F81BD', '#4F81BD']
)

table.auto_set_fontsize(False)
table.set_fontsize(10)
table.scale(1.2, 1.4)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.get_text().set_color('white')
        cell.get_text().set_weight('bold')
    elif row % 2 == 0:
        cell.set_facecolor('#F2F2F2')

plt.title("Model Comparison: Custom CNN vs MobileNetV2", fontsize=12, pad=15, weight='bold')
plt.savefig('outputs/figures/comparison_table.png', bbox_inches='tight')
plt.show()